# 00 · Training Zip Structure Check

**Project:** Physics-Informed Trigger Event Analysis — Urban Intersection Safety  
**Dataset:** USDOT Intersection Safety Challenge Stage-1B  
**Stage:** Task 0 — zip inventory (no extraction, no ML)

---

## What this notebook does

1. Clones the project repo from GitHub into Colab `/content/`.
2. Mounts Google Drive **read-only** to access the training zip files.
3. Inspects each zip using Python `zipfile` **without extracting** anything.
4. Counts files, extensions, unique `Run_XXXX` folders, and named file patterns.
5. Probes a small sample of `traffic-triggers-output.json` files for structure.
6. Saves four CSV outputs to `outputs/tables/` inside `/content/` (not Drive).
7. Zips all outputs and triggers a browser download — push the zip to GitHub manually.

## What this notebook does NOT do

- Does **not** extract any file to disk.
- Does **not** read CSV or radar JSON contents.
- Does **not** process video or pcap files.
- Does **not** train any model.

## Outputs

| File | Description |
|------|-------------|
| `outputs/tables/training_zip_inventory.csv` | One row per zip — counts and flags |
| `outputs/tables/training_zip_sample_paths.csv` | Sampled internal paths (≤ MAX_SAMPLE_PATHS per zip) |
| `outputs/tables/sample_trigger_semantics_probe.csv` | Trigger JSON structure summary |
| `outputs/tables/sample_trigger_reference_names.csv` | Unique trigger reference names found |

---
## 0 · Optional: Install / Verify Dependencies

Run only if `pandas` or `tqdm` are missing in your Colab environment.

In [1]:
# Uncomment the line below only if packages are not available.
# !pip install -q pandas tqdm
print("[OK] Dependency check cell ready — uncomment pip install if needed.")

[OK] Dependency check cell ready — uncomment pip install if needed.


---
## 1 · Clone Repo from GitHub

This cell clones the project repo into Colab's `/content/` directory so that
the `src/` modules are available for import.

**Edit `GITHUB_REPO_URL` to match your repository before running.**

If the repo is already cloned (e.g. you re-opened the notebook), the clone
step is skipped automatically.

In [2]:
import os, subprocess

# ╔══════════════════════════════════════════════════════════════════╗
# ║  EDIT THIS URL to match your GitHub repository                 ║
# ╠══════════════════════════════════════════════════════════════════╣
GITHUB_REPO_URL = "https://github.com/PulockDas/intersection_safety_trigger_project.git"
# ╚══════════════════════════════════════════════════════════════════╝

CLONE_DIR = "/content/intersection_safety_trigger_project"

if not os.path.exists(CLONE_DIR):
    print(f"[INFO] Cloning {GITHUB_REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO_URL, CLONE_DIR],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("[OK]  Clone successful.")
    else:
        print("[ERROR] Clone failed:")
        print(result.stderr)
        raise RuntimeError("Git clone failed — check GITHUB_REPO_URL and try again.")
else:
    print(f"[INFO] Repo already exists at {CLONE_DIR} — skipping clone.")
    print("       To re-clone: Runtime > Disconnect and delete runtime, then re-run.")

print(f"\n[INFO] Repo contents:")
for item in sorted(os.listdir(CLONE_DIR)):
    print(f"  {item}")

[INFO] Cloning https://github.com/PulockDas/intersection_safety_trigger_project.git ...
[OK]  Clone successful.

[INFO] Repo contents:
  .git
  .gitignore
  README.md
  notebooks
  requirements.txt
  src


### 1.1 · Mount Google Drive (read-only — for training zip files)

Drive is mounted **only to read the training zip files**.
No outputs are saved to Drive.
You will set `TRAINING_ZIP_DIR` in the next section to point at your zip files.

In [3]:
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("[INFO] Google Drive mounted at /content/drive")
    print("       Drive is used ONLY to read training zip files.")
    print("       All outputs are saved to /content/ (not Drive).")
except ImportError:
    IN_COLAB = False
    print("[INFO] Not running in Colab — Drive mount skipped.")
    print("       Set TRAINING_ZIP_DIR manually in the next cell.")

Mounted at /content/drive
[INFO] Google Drive mounted at /content/drive
       Drive is used ONLY to read training zip files.
       All outputs are saved to /content/ (not Drive).


---
## 2 · Project Setup

`PROJECT_ROOT` is fixed to the cloned repo at `/content/intersection_safety_trigger_project` — **do not change it**.

### About shared-folder shortcuts

If your training data lives in a **shared folder** (someone shared it with you
and you added a shortcut to your Drive), the folder may **not** be reachable at
`MyDrive/<name>` in Colab — shortcuts are not always followed.

Run **Cell 2.0** first. It scans both `MyDrive/` and `Shareddrives/` and
auto-detects the correct path for you.  
If auto-detection fails, it will print exactly what is visible so you can
set the path manually in **Cell 2.1**.

### 2.0 · Locate the Training Data Folder

In [4]:
import os
from pathlib import Path

# ╔══════════════════════════════════════════════════════════════════╗
# ║  Set the name of your shared / shortcut folder here            ║
# ╠══════════════════════════════════════════════════════════════════╣
FOLDER_NAME = "ITS_Intersection_USDOT"
# ╚══════════════════════════════════════════════════════════════════╝

DRIVE_ROOT   = Path("/content/drive")
MY_DRIVE     = DRIVE_ROOT / "MyDrive"
SHARED_DRIVE = DRIVE_ROOT / "Shareddrives"

# ── Print Drive structure for debugging ─────────────────────────────────────
print("Contents of /content/drive/:")
if DRIVE_ROOT.exists():
    for item in sorted(DRIVE_ROOT.iterdir()):
        print(f"  {item.name}/")
else:
    print("  [Drive not mounted — run Section 1.1 first]")

print()
print("Top-level items in MyDrive/ (first 40):")
if MY_DRIVE.exists():
    for item in sorted(MY_DRIVE.iterdir())[:40]:
        marker = "/" if item.is_dir() else ""
        print(f"  {item.name}{marker}")
else:
    print("  [MyDrive not accessible]")

print()
print("Shared Drives:")
if SHARED_DRIVE.exists():
    items = list(SHARED_DRIVE.iterdir())
    if items:
        for item in sorted(items):
            print(f"  {item.name}/")
    else:
        print("  [no shared drives found]")
else:
    print("  [Shareddrives/ not present]")

# ── Try candidate paths in priority order ────────────────────────────────────
candidates = [
    MY_DRIVE     / FOLDER_NAME,
    SHARED_DRIVE / FOLDER_NAME,
    MY_DRIVE     / "My Drive" / FOLDER_NAME,   # rare alternative mount
]

TRAINING_ZIP_DIR = None
print()
print(f"Searching for '{FOLDER_NAME}' ...")
for c in candidates:
    try:
        if c.exists() and c.is_dir():
            zip_count = len(list(c.glob("*.zip")))
            print(f"  [FOUND] {c}  ({zip_count} zip file(s))")
            if zip_count > 0 and TRAINING_ZIP_DIR is None:
                TRAINING_ZIP_DIR = c
    except PermissionError:
        print(f"  [PERMISSION ERROR] {c}")

print()
if TRAINING_ZIP_DIR:
    print(f"[OK] Auto-detected TRAINING_ZIP_DIR = {TRAINING_ZIP_DIR}")
else:
    print(f"[WARN] Could not auto-detect '{FOLDER_NAME}'.")
    print("       Check the listing above and set the path manually in Cell 2.1.")

Contents of /content/drive/:
  .Encrypted/
  .Trash-0/
  .shortcut-targets-by-id/
  MyDrive/
  Othercomputers/

Top-level items in MyDrive/ (first 40):
  0_Introduction.gdoc
  2 Pdf_12_09_11_24_09.pdf
  2017831029.gdoc
  2017831036.gdoc
  2017831036_Verse.pptx
  2017831044_B.gdoc
  2017831046/
  2017831046 (1).pdf
  2017831046 (2).pdf
  2017831046 (3).pdf
  2017831046 (4).pdf
  2017831046 (5).pdf
  2017831046 (6).pdf
  2017831046 (7).pdf
  2017831046 (8).pdf
  2017831046.gdoc
  2017831046.pdf
  2017831046_A.pdf
  2017831046_B.pdf
  2017831046_Internship Report PSL.pdf
  20180914_115455.jpg
  20180914_115631.jpg
  20180914_115745.jpg
  20180914_115749.jpg
  20180914_132000.jpg
  20180914_132003.jpg
  20180914_132103.jpg
  20180914_132111.jpg
  20180914_133643.jpg
  20180914_145911.jpg
  20180914_151243.jpg
  20180914_151413 (1).jpg
  20180914_151413.jpg
  20181011_125902.jpg
  20190129_220734.jpg
  20190314_160847.jpg
  20190314_160849.jpg
  20190314_160852.jpg
  20190314_173758.jpg
  2

### 2.1 · Set Paths

If auto-detection above succeeded, this cell uses that result automatically.
If it failed, uncomment the **manual override** line and paste in the correct path
from the listing printed above.

In [5]:
from pathlib import Path
import sys

# PROJECT_ROOT is always the cloned repo — do not edit this line.
PROJECT_ROOT = Path('/content/intersection_safety_trigger_project')

# ── Manual override (uncomment and edit if Cell 2.0 auto-detection failed) ──
# TRAINING_ZIP_DIR = Path('/content/drive/MyDrive/ITS_Intersection_USDOT')

# Confirm TRAINING_ZIP_DIR was resolved (either auto or manual)
if 'TRAINING_ZIP_DIR' not in dir() or TRAINING_ZIP_DIR is None:
    raise RuntimeError(
        "TRAINING_ZIP_DIR is not set.\n"
        "Uncomment and edit the manual override line above."
    )

# Add src/ to sys.path so project modules can be imported
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Outputs live in /content/ — NOT in Google Drive
OUTPUTS_DIR  = PROJECT_ROOT / 'outputs'
TABLES_DIR   = OUTPUTS_DIR  / 'tables'
FIGURES_DIR  = OUTPUTS_DIR  / 'figures'
SAMPLES_DIR  = OUTPUTS_DIR  / 'samples'
LOGS_DIR     = OUTPUTS_DIR  / 'logs'

print(f"PROJECT_ROOT     = {PROJECT_ROOT}")
print(f"SRC_DIR exists   = {SRC_DIR.exists()}")
print(f"TRAINING_ZIP_DIR = {TRAINING_ZIP_DIR}")
print(f"OUTPUTS_DIR      = {OUTPUTS_DIR}  [/content/ — not Drive]")

PROJECT_ROOT     = /content/intersection_safety_trigger_project
SRC_DIR exists   = True
TRAINING_ZIP_DIR = /content/drive/MyDrive/ITS_Intersection_USDOT
OUTPUTS_DIR      = /content/intersection_safety_trigger_project/outputs  [/content/ — not Drive]


### 2.2 · Imports

In [6]:
import json
import zipfile
import re
from collections import Counter
from pathlib import Path

import pandas as pd

# ── Project modules ──────────────────────────────────────────────────────────
from config import (
    FILE_PATTERNS,
    RUN_PATTERN,
    MAX_SAMPLE_PATHS,
    MAX_TRIGGER_FILES_TO_INSPECT,
    EXPECTED_LABELS,
    HEAVY_FILE_LABELS,
)
from file_discovery import find_training_zips, make_output_dirs
from zip_utils import scan_zip, read_json_from_zip, probe_trigger_json
from utils import bytes_to_gb, format_size, section, check_mark, yes_no

print("[OK] All imports successful.")
print(f"     MAX_SAMPLE_PATHS             = {MAX_SAMPLE_PATHS}")
print(f"     MAX_TRIGGER_FILES_TO_INSPECT = {MAX_TRIGGER_FILES_TO_INSPECT}")

[OK] All imports successful.
     MAX_SAMPLE_PATHS             = 200
     MAX_TRIGGER_FILES_TO_INSPECT = 3


### 2.3 · Create Output Directories

In [7]:
make_output_dirs(TABLES_DIR, FIGURES_DIR, SAMPLES_DIR, LOGS_DIR)
print("[OK] Output directories are ready.")

[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/tables
[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/figures
[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/samples
[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/logs
[OK] Output directories are ready.


---
## 3 · Discover Training Zip Files

In [8]:
zip_paths = find_training_zips(TRAINING_ZIP_DIR)

if not zip_paths:
    raise FileNotFoundError(
        f"No zip files found in {TRAINING_ZIP_DIR}.\n"
        "Check TRAINING_ZIP_DIR in Cell 2 and re-run."
    )

print("\n── Zip files discovered ─────────────────────────────────────────")
total_raw_bytes = 0
for i, zp in enumerate(zip_paths, 1):
    size_bytes = zp.stat().st_size
    total_raw_bytes += size_bytes
    print(f"  [{i}] {zp.name:<50s}  {format_size(size_bytes):>10s}")

print(f"\n  Total compressed size: {format_size(total_raw_bytes)}")

[INFO]  Found 4 zip file(s) in: /content/drive/MyDrive/ITS_Intersection_USDOT

── Zip files discovered ─────────────────────────────────────────
  [1] Training Data 1.zip                                  151.06 GB
  [2] Training Data 2.zip                                  150.50 GB
  [3] Training Data 3.zip                                  152.57 GB
  [4] Training Data 4.zip                                  149.71 GB

  Total compressed size: 603.85 GB


---
## 4 · Scan Zip Contents

Each zip's central directory is read to count files, extensions, Run IDs,
and named file patterns.  **No files are extracted.**

This may take 1–3 minutes per zip depending on the number of entries
and Drive I/O speed.

In [9]:
all_scans: list[dict] = []

for zip_path in zip_paths:
    print(f"\n{'─'*60}")
    print(f"Scanning: {zip_path.name}")
    scan = scan_zip(zip_path, max_sample_paths=MAX_SAMPLE_PATHS)
    all_scans.append(scan)

    print(f"  Total files    : {scan['total_files']:,}")
    print(f"  Unique Run IDs : {len(scan['run_ids']):,}")
    print(f"  Trigger files  : {scan['pattern_counts']['traffic_trigger_json']:,}")
    print(f"  Top extensions : {dict(scan['extension_counts'].most_common(6))}")

print(f"\n{'─'*60}")
print(f"[OK] All {len(all_scans)} zips scanned.")


────────────────────────────────────────────────────────────
Scanning: Training Data 1.zip
  [scan] Opening Training Data 1.zip …
  [scan] Found 6,748 internal entries — scanning …
  [scan] Done — 200 unique Run IDs detected
  Total files    : 6,748
  Unique Run IDs : 200
  Trigger files  : 104
  Top extensions : {'.csv': 3083, '.mp4': 2600, '.pcap': 540, '.json': 525}

────────────────────────────────────────────────────────────
Scanning: Training Data 2.zip
  [scan] Opening Training Data 2.zip …
  [scan] Found 6,730 internal entries — scanning …
  [scan] Done — 200 unique Run IDs detected
  Total files    : 6,730
  Unique Run IDs : 200
  Trigger files  : 99
  Top extensions : {'.csv': 3090, '.mp4': 2600, '.pcap': 545, '.json': 495}

────────────────────────────────────────────────────────────
Scanning: Training Data 3.zip
  [scan] Opening Training Data 3.zip …
  [scan] Found 6,732 internal entries — scanning …
  [scan] Done — 200 unique Run IDs detected
  Total files    : 6,732
  Un

### 4.1 · Per-zip Pattern Summary

In [10]:
print(f"{'Zip':<35s}  ", end="")
for label in FILE_PATTERNS:
    print(f"{label[:18]:<19s}", end="")
print()
print("-" * (35 + 2 + 19 * len(FILE_PATTERNS)))

for scan in all_scans:
    print(f"{scan['zip_name'][:34]:<35s}  ", end="")
    for label in FILE_PATTERNS:
        cnt = scan['pattern_counts'][label]
        print(f"{cnt:<19d}", end="")
    print()

Zip                                  gt_csv             radar_sensor_json  traffic_trigger_js v2xhub_csv         visual_camera_timi thermal_camera_tim isc_timing_csv     v2xhub_timing_csv  labels_csv         video_file         pcap_file          
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Training Data 1.zip                  4                  416                104                139                1591               995                199                139                0                  2600               540                
Training Data 2.zip                  0                  396                99                 145                1600               1000               200                145                0                  2600               545                
Training Dat

---
## 5 · Save Inventory & Sample Paths

### 5.1 · Build and Save Inventory CSV

One row per zip, columns for size, file counts, run ID count, and all
named pattern counts.

In [11]:
inventory_rows = []

for scan in all_scans:
    row = {
        "zip_name":         scan["zip_name"],
        "zip_size_gb":      scan["zip_size_gb"],
        "total_files":      scan["total_files"],
        "unique_run_ids":   len(scan["run_ids"]),
        "run_id_list":      "|".join(scan["run_ids"]),
    }
    # Named pattern counts
    for label in FILE_PATTERNS:
        row[f"count_{label}"] = scan["pattern_counts"][label]
    # Top-5 extensions
    row["top_extensions"] = str(
        dict(scan["extension_counts"].most_common(5))
    )
    inventory_rows.append(row)

inventory_df = pd.DataFrame(inventory_rows)

out_path = TABLES_DIR / "training_zip_inventory.csv"
inventory_df.to_csv(out_path, index=False)
print(f"[SAVED] {out_path}")
print(f"        Shape: {inventory_df.shape}")
inventory_df

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/training_zip_inventory.csv
        Shape: (4, 17)


,zip_name,zip_size_gb,total_files,unique_run_ids,run_id_list,count_gt_csv,count_radar_sensor_json,count_traffic_trigger_json,count_v2xhub_csv,count_visual_camera_timing,count_thermal_camera_timing,count_isc_timing_csv,count_v2xhub_timing_csv,count_labels_csv,count_video_file,count_pcap_file,top_extensions
0,Training Data 1.zip,151.061,6748,200,0003|0026|0048|0055|0063|0068|0078|0083|0086|0...,4,416,104,139,1591,995,199,139,0,2600,540,"{'.csv': 3083, '.mp4': 2600, '.pcap': 540, '.j..."
1,Training Data 2.zip,150.501,6730,200,0019|0021|0032|0033|0034|0045|0057|0064|0066|0...,0,396,99,145,1600,1000,200,145,0,2600,545,"{'.csv': 3090, '.mp4': 2600, '.pcap': 545, '.j..."
2,Training Data 3.zip,152.574,6732,200,0001|0008|0011|0013|0024|0029|0030|0037|0039|0...,0,400,100,144,1600,1000,200,144,0,2600,544,"{'.csv': 3088, '.mp4': 2600, '.pcap': 544, '.j..."
3,Training Data 4.zip,149.714,6654,200,0006|0007|0010|0016|0025|0038|0044|0049|0052|0...,0,352,88,138,1600,1000,200,138,0,2600,538,"{'.csv': 3076, '.mp4': 2600, '.pcap': 538, '.j..."


### 5.2 · Save Sample Internal Paths CSV

In [12]:
sample_rows = []
for scan in all_scans:
    for path_str in scan["sample_paths"]:
        # Identify which pattern(s) this path matches
        matched = [
            label for label, regex in FILE_PATTERNS.items()
            if regex.search(path_str)
        ]
        m = RUN_PATTERN.search(path_str)
        sample_rows.append({
            "zip_name":    scan["zip_name"],
            "internal_path": path_str,
            "run_id":      m.group(1).zfill(4) if m else "",
            "extension":   Path(path_str).suffix.lower(),
            "matched_patterns": "|".join(matched) if matched else "",
        })

sample_paths_df = pd.DataFrame(sample_rows)

out_path = TABLES_DIR / "training_zip_sample_paths.csv"
sample_paths_df.to_csv(out_path, index=False)
print(f"[SAVED] {out_path}")
print(f"        Shape: {sample_paths_df.shape}")
sample_paths_df.head(15)

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/training_zip_sample_paths.csv
        Shape: (800, 5)


,zip_name,internal_path,run_id,extension,matched_patterns
0,Training Data 1.zip,Run_166/VisualCamera3_Run_166.mp4,0166,.mp4,video_file
1,Training Data 1.zip,Run_166/Radars_Run_166_traffic-triggers-output...,0166,.json,traffic_trigger_json
2,Training Data 1.zip,Run_166/Radars_Run_166_sensor1.json,0166,.json,radar_sensor_json
3,Training Data 1.zip,Run_166/VisualCamera1_Run_166.mp4,0166,.mp4,video_file
4,Training Data 1.zip,Run_166/ThermalCamera5_Run_166.mp4,0166,.mp4,video_file
5,Training Data 1.zip,Run_166/ThermalCamera3_Run_166.mp4,0166,.mp4,video_file
6,Training Data 1.zip,Run_166/VisualCamera8_Run_166.mp4,0166,.mp4,video_file
7,Training Data 1.zip,Run_166/VisualCamera7_Run_166.mp4,0166,.mp4,video_file
8,Training Data 1.zip,Run_166/Radars_Run_166_sensor4.json,0166,.json,radar_sensor_json
9,Training Data 1.zip,Run_166/Lidar1_Run_166.pcap,0166,.pcap,pcap_file


---
## 6 · Trigger Semantics Probe

> **Semantic Warning:**  
> Traffic trigger `reference_name` values are **not yet semantically decoded**.
> At this stage, they should be treated as **detector/zone activation
> identifiers**, not confirmed safety event categories.

---

### Current Working Interpretation

`traffic-triggers-output.json` likely represents processed traffic detector
or zone activation events. Each trigger appears linked to a lane, sensor,
zone, and `reference_name`. However, the **exact semantic meaning** of each
`reference_name` must be inferred later using metadata, lane/zone mapping,
timing behaviour, radar object movement, and possibly nearby camera frames.

---

### TODO — Later Semantic Analysis

Later notebooks should:
- Count how often each trigger `reference_name` appears across all runs.
- Map `reference_name` → lane, zone, and sensor.
- Check whether each trigger activates during object presence in that zone.
- Compare trigger timestamps with radar object positions.
- Determine if triggers correspond to: vehicle arrival, lane occupancy,
  zone entry, queue activity, or conflict-related behaviour.
- Use camera timing files to retrieve nearby frames for selected examples.
- **Avoid claiming exact trigger meaning without cross-modal evidence.**

### 6.1 · Identify Trigger Files Across All Zips

In [13]:
print("── Trigger file inventory ──────────────────────────────────────")
for scan in all_scans:
    trigger_paths = scan["trigger_file_paths"]
    print(f"\n  {scan['zip_name']}")
    print(f"    Trigger files found : {len(trigger_paths):,}")
    if trigger_paths:
        # Unique run IDs that have a trigger file
        trigger_run_ids = sorted(set(
            m.group(1).zfill(4)
            for p in trigger_paths
            for m in [RUN_PATTERN.search(p)] if m
        ))
        print(f"    Runs with triggers  : {len(trigger_run_ids)}")
        print(f"    Sample paths:")
        for p in trigger_paths[:3]:
            print(f"      {p}")
        if len(trigger_paths) > 3:
            print(f"      … and {len(trigger_paths) - 3} more")

── Trigger file inventory ──────────────────────────────────────

  Training Data 1.zip
    Trigger files found : 104
    Runs with triggers  : 104
    Sample paths:
      Run_166/Radars_Run_166_traffic-triggers-output.json
      Run_844/Radars_Run_844_traffic-triggers-output.json
      Run_327/Radars_Run_327_traffic-triggers-output.json
      … and 101 more

  Training Data 2.zip
    Trigger files found : 99
    Runs with triggers  : 99
    Sample paths:
      Run_324/Radars_Run_324_traffic-triggers-output.json
      Run_484/Radars_Run_484_traffic-triggers-output.json
      Run_211/Radars_Run_211_traffic-triggers-output.json
      … and 96 more

  Training Data 3.zip
    Trigger files found : 100
    Runs with triggers  : 100
    Sample paths:
      Run_89/Radars_Run_89_traffic-triggers-output.json
      Run_405/Radars_Run_405_traffic-triggers-output.json
      Run_591/Radars_Run_591_traffic-triggers-output.json
      … and 97 more

  Training Data 4.zip
    Trigger files found : 88
 

### 6.2 · Probe Sampled Trigger JSON Files

For up to `MAX_TRIGGER_FILES_TO_INSPECT` trigger JSON files across all zips,
read the file **directly from inside the zip** (no extraction) and inspect
its structure.

In [14]:
probe_results: list[dict] = []
inspected_total = 0

print(f"Probing up to {MAX_TRIGGER_FILES_TO_INSPECT} trigger files total …\n")

for scan in all_scans:
    zip_path = TRAINING_ZIP_DIR / scan["zip_name"]
    trigger_paths = scan["trigger_file_paths"]

    if not trigger_paths:
        print(f"  [SKIP] {scan['zip_name']} — no trigger files")
        continue

    # How many slots remain?
    slots = MAX_TRIGGER_FILES_TO_INSPECT - inspected_total
    if slots <= 0:
        print(f"  [SKIP] {scan['zip_name']} — probe quota reached")
        continue

    to_probe = trigger_paths[:slots]
    print(f"  {scan['zip_name']} → probing {len(to_probe)} file(s)")

    for tp in to_probe:
        print(f"    Reading: {tp}")
        result = probe_trigger_json(zip_path, tp)
        probe_results.append(result)
        inspected_total += 1

        print(f"      top_level_type  : {result['top_level_type']}")
        print(f"      top_level_keys  : {result['top_level_keys']}")
        print(f"      first_item_keys : {result['first_item_keys']}")
        print(f"      ref names found : {result['sample_reference_names']}")
        print(f"      lanes found     : {result['sample_associated_lanes']}")
        print(f"      zones found     : {result['sample_associated_zones']}")
        if result["error"]:
            print(f"      ERROR           : {result['error']}")

print(f"\n[OK] Probed {inspected_total} trigger file(s).")

Probing up to 3 trigger files total …

  Training Data 1.zip → probing 3 file(s)
    Reading: Run_166/Radars_Run_166_traffic-triggers-output.json
      top_level_type  : list
      top_level_keys  : ['<list>']
      first_item_keys : ['topic', 'payload', 'qos', 'receivedAt', 'retain']
      ref names found : []
      lanes found     : []
      zones found     : []
    Reading: Run_844/Radars_Run_844_traffic-triggers-output.json
      top_level_type  : list
      top_level_keys  : ['<list>']
      first_item_keys : ['topic', 'payload', 'qos', 'receivedAt', 'retain']
      ref names found : []
      lanes found     : []
      zones found     : []
    Reading: Run_327/Radars_Run_327_traffic-triggers-output.json
      top_level_type  : list
      top_level_keys  : ['<list>']
      first_item_keys : ['topic', 'payload', 'qos', 'receivedAt', 'retain']
      ref names found : []
      lanes found     : []
      zones found     : []
  [SKIP] Training Data 2.zip — probe quota reached
  [SKIP] T

### 6.3 · Save Trigger Semantics Probe CSV

In [15]:
if probe_results:
    # Flatten list fields to pipe-delimited strings for CSV
    flat_rows = []
    for r in probe_results:
        flat_rows.append({
            "zip_name":                  r["zip_name"],
            "internal_path":             r["path"],
            "top_level_type":            r["top_level_type"],
            "top_level_keys":            "|".join(r["top_level_keys"]),
            "first_item_keys":           "|".join(r["first_item_keys"]),
            "sample_reference_names":    "|".join(r["sample_reference_names"]),
            "sample_associated_lanes":   "|".join(r["sample_associated_lanes"]),
            "sample_associated_zones":   "|".join(r["sample_associated_zones"]),
            "sample_associated_sensors": "|".join(r["sample_associated_sensors"]),
            "sample_timestamps":         "|".join(r["sample_timestamps"]),
            "error":                     r["error"] or "",
        })

    probe_df = pd.DataFrame(flat_rows)
    out_path = TABLES_DIR / "sample_trigger_semantics_probe.csv"
    probe_df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path}")
    print(f"        Shape: {probe_df.shape}")
    display(probe_df)
else:
    print("[WARN] No trigger files were probed. Skipping save.")
    probe_df = pd.DataFrame()

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/sample_trigger_semantics_probe.csv
        Shape: (3, 11)


,zip_name,internal_path,top_level_type,top_level_keys,first_item_keys,sample_reference_names,sample_associated_lanes,sample_associated_zones,sample_associated_sensors,sample_timestamps,error
0,Training Data 1.zip,Run_166/Radars_Run_166_traffic-triggers-output...,list,<list>,topic|payload|qos|receivedAt|retain,,,,,2024-02-16T14:55:51+00:00,
1,Training Data 1.zip,Run_844/Radars_Run_844_traffic-triggers-output...,list,<list>,topic|payload|qos|receivedAt|retain,,,,,2024-01-31T15:37:11+00:00,
2,Training Data 1.zip,Run_327/Radars_Run_327_traffic-triggers-output...,list,<list>,topic|payload|qos|receivedAt|retain,,,,,2024-01-30T00:42:36+00:00|2024-01-30T00:42:37+...,


### 6.4 · Save Unique Trigger Reference Names CSV

In [16]:
all_ref_names: list[str] = []
for r in probe_results:
    all_ref_names.extend(r["sample_reference_names"])

unique_ref_names = sorted(set(all_ref_names))

if unique_ref_names:
    ref_df = pd.DataFrame({
        "reference_name": unique_ref_names,
        "note": (
            "Sampled from trigger JSON probe — semantic meaning not yet decoded"
        ),
    })
    out_path = TABLES_DIR / "sample_trigger_reference_names.csv"
    ref_df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path}")
    print(f"        {len(unique_ref_names)} unique reference name(s) found:")
    for n in unique_ref_names:
        print(f"          {n}")
else:
    print("[WARN] No reference names found in probed files.")
    print("       This may be because:")
    print("         1. Trigger files were not found in the zips.")
    print("         2. The JSON structure uses different field names.")
    print("         3. MAX_TRIGGER_FILES_TO_INSPECT is 0.")

print()
print("⚠  SEMANTIC WARNING:")
print("   Traffic trigger reference names are not yet semantically decoded.")
print("   At this stage, they should be treated as detector/zone activation")
print("   identifiers, not confirmed safety event categories.")

[WARN] No reference names found in probed files.
       This may be because:
         1. Trigger files were not found in the zips.
         2. The JSON structure uses different field names.
         3. MAX_TRIGGER_FILES_TO_INSPECT is 0.

⚠  SEMANTIC WARNING:
   Traffic trigger reference names are not yet semantically decoded.
   At this stage, they should be treated as detector/zone activation
   identifiers, not confirmed safety event categories.


---
## 7 · Final Summary Report

Aggregated findings across all 4 training zip files.

In [17]:
# ── Aggregate across all scans ───────────────────────────────────────────────
total_zip_count    = len(all_scans)
total_size_gb      = sum(s["zip_size_gb"] for s in all_scans)
total_files        = sum(s["total_files"] for s in all_scans)
all_run_ids        = sorted(set(rid for s in all_scans for rid in s["run_ids"]))
total_unique_runs  = len(all_run_ids)

# Aggregate pattern counts
agg_patterns: dict[str, int] = {k: 0 for k in FILE_PATTERNS}
for scan in all_scans:
    for label, cnt in scan["pattern_counts"].items():
        agg_patterns[label] += cnt

# ── Missing file warnings ────────────────────────────────────────────────────
missing_labels = [
    label for label in EXPECTED_LABELS
    if agg_patterns.get(label, 0) == 0
]

# ── Print summary ────────────────────────────────────────────────────────────
sep = "═" * 62
print(sep)
print("  TRAINING ZIP STRUCTURE CHECK — FINAL SUMMARY")
print(sep)
print(f"  Zip files found          : {total_zip_count}")
print(f"  Total compressed size    : {total_size_gb:.2f} GB")
print(f"  Total internal files     : {total_files:,}")
print(f"  Unique Run IDs discovered: {total_unique_runs:,}")
print()
print("  File type presence:")
print(f"    GT CSV files           : {yes_no(agg_patterns['gt_csv'] > 0)}"
      f"  ({agg_patterns['gt_csv']:,})")
print(f"    Radar sensor JSON      : {yes_no(agg_patterns['radar_sensor_json'] > 0)}"
      f"  ({agg_patterns['radar_sensor_json']:,})")
print(f"    Traffic trigger JSON   : {yes_no(agg_patterns['traffic_trigger_json'] > 0)}"
      f"  ({agg_patterns['traffic_trigger_json']:,})")
print(f"    V2X hub CSV            : {yes_no(agg_patterns['v2xhub_csv'] > 0)}"
      f"  ({agg_patterns['v2xhub_csv']:,})")
print(f"    Visual camera timing   : {yes_no(agg_patterns['visual_camera_timing'] > 0)}"
      f"  ({agg_patterns['visual_camera_timing']:,})")
print(f"    Thermal camera timing  : {yes_no(agg_patterns['thermal_camera_timing'] > 0)}"
      f"  ({agg_patterns['thermal_camera_timing']:,})")
print(f"    ISC timing CSV         : {yes_no(agg_patterns['isc_timing_csv'] > 0)}"
      f"  ({agg_patterns['isc_timing_csv']:,})")
print(f"    Video files            : {yes_no(agg_patterns['video_file'] > 0)}"
      f"  ({agg_patterns['video_file']:,})  [NOT processed — heavy]")
print(f"    PCAP files             : {yes_no(agg_patterns['pcap_file'] > 0)}"
      f"  ({agg_patterns['pcap_file']:,})  [NOT processed — heavy]")
print()
print("  Trigger semantics probe:")
print(f"    Files probed           : {inspected_total}")
print(f"    Unique ref names found : {len(unique_ref_names)}")
print()

if missing_labels:
    print("  ⚠  MISSING FILE WARNINGS:")
    for label in missing_labels:
        print(f"       {label} — count is 0 across all zips")
    print("     This could mean the files don't exist in the dataset,")
    print("     or the filename pattern in config.py needs adjustment.")
else:
    print("  ✓  All expected file types detected.")

print()
print("  Outputs saved to outputs/tables/:")
for csv_name in [
    "training_zip_inventory.csv",
    "training_zip_sample_paths.csv",
    "sample_trigger_semantics_probe.csv",
    "sample_trigger_reference_names.csv",
]:
    fpath = TABLES_DIR / csv_name
    exists = fpath.exists()
    print(f"    {check_mark(exists)} {csv_name}")

print()
print(sep)
print("  Task 0 complete. Next: Notebook 01 — Radar JSON Schema Inspection.")
print(sep)

══════════════════════════════════════════════════════════════
  TRAINING ZIP STRUCTURE CHECK — FINAL SUMMARY
══════════════════════════════════════════════════════════════
  Zip files found          : 4
  Total compressed size    : 603.85 GB
  Total internal files     : 26,864
  Unique Run IDs discovered: 800

  File type presence:
    GT CSV files           : YES  (4)
    Radar sensor JSON      : YES  (1,564)
    Traffic trigger JSON   : YES  (391)
    V2X hub CSV            : YES  (566)
    Visual camera timing   : YES  (6,391)
    Thermal camera timing  : YES  (3,995)
    ISC timing CSV         : YES  (799)
    Video files            : YES  (10,400)  [NOT processed — heavy]
    PCAP files             : YES  (2,167)  [NOT processed — heavy]

  Trigger semantics probe:
    Files probed           : 3
    Unique ref names found : 0

  ✓  All expected file types detected.

  Outputs saved to outputs/tables/:
    ✓ training_zip_inventory.csv
    ✓ training_zip_sample_paths.csv
    ✓ samp

---
## 8 · Package Outputs and Download

All four CSVs are zipped into a single archive inside `/content/`.
The next cell triggers a **browser download** so you can save the zip locally,
add it to the repo, and push to GitHub.

Workflow after download:
```
1. Unzip and copy the CSVs into  outputs/tables/
2. git add outputs/tables/*.csv
3. git commit -m "add Task 0 output CSVs"
4. git push
```

In [18]:
import shutil, datetime
from pathlib import Path

# Build a timestamped archive name
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_stem  = f"task0_outputs_{timestamp}"
zip_path  = Path(f"/content/{zip_stem}.zip")

# Zip the entire outputs/ directory
shutil.make_archive(
    str(zip_path.with_suffix("")),  # archive path without .zip extension
    "zip",                          # format
    str(OUTPUTS_DIR),               # root directory to compress
)

print(f"[OK] Archive created: {zip_path}")
print(f"     Size: {zip_path.stat().st_size / 1024:.1f} KB")
print()
print("Contents included:")
for f in sorted(TABLES_DIR.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  tables/{f.name}  ({size_kb:.1f} KB)")

[OK] Archive created: /content/task0_outputs_20260514_025519.zip
     Size: 7.6 KB

Contents included:
  tables/sample_trigger_semantics_probe.csv  (0.7 KB)
  tables/training_zip_inventory.csv  (4.7 KB)
  tables/training_zip_sample_paths.csv  (66.9 KB)


In [19]:
# Trigger browser download.
# A 'Save file' dialog will appear. If it doesn't, check your pop-up blocker.
# You can also download manually from the Colab file browser (left panel).
try:
    from google.colab import files
    print(f"[INFO] Starting download: {zip_path.name}")
    files.download(str(zip_path))
    print("[OK]  Download initiated — check your browser downloads folder.")
except ImportError:
    print(f"[INFO] Not in Colab.")
    print(f"       The archive is at: {zip_path}")

[INFO] Starting download: task0_outputs_20260514_025519.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[OK]  Download initiated — check your browser downloads folder.


---

## Notebook Complete

### Files saved (in `/content/intersection_safety_trigger_project/outputs/tables/`)

| Output | Description |
|--------|-------------|
| `training_zip_inventory.csv` | One row per zip — size, counts, run IDs |
| `training_zip_sample_paths.csv` | Sampled internal paths with pattern labels |
| `sample_trigger_semantics_probe.csv` | Trigger JSON structure from sampled files |
| `sample_trigger_reference_names.csv` | Unique trigger reference names found |

A zip of all outputs was downloaded via Section 8.  
Push it (or the extracted CSVs) to GitHub to preserve results.

### Next Steps

- **Notebook 01**: Inspect the schema of `Radars_Run_XXXX_sensor1.json` — field names, coordinate system, object IDs, timestamps.
- **Notebook 02**: Inspect `Radars_Run_XXXX_traffic-triggers-output.json` in detail — begin semantic decoding of `reference_name` values.
- **Notebook 03**: Build the trigger vs. non-trigger window dataset from radar detections and trigger timestamps.

### Semantic Reminder

> Trigger `reference_name` values discovered in this notebook are **preliminary
> structural observations only**. Do not interpret them as confirmed event
> categories until cross-modal validation (Notebook 02) is complete.